In [1]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import gdown

# Unzip the downloaded data
zip_file_path = './keyword_data.zip'
extract_path = './keyword_spot/'
data_path = './keyword_spot/keyword_data/'

if not os.path.exists(zip_file_path):
  !gdown 1bG3WF61-26QrINZ0_KRqPXmWAkCycBV1

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f'Data extracted to: {extract_path}')

Data extracted to: ./keyword_spot/


After extracting the data, we'll create a list of all audio file paths and their corresponding labels (up, down, left, right). We'll then convert this into a pandas DataFrame for easier management.

In [2]:
NUM_CLASSES = 5
TRAIN = False
TUNE_FE = True
TUNE_MODEL = False

In [3]:
# Collect all audio file paths and their labels
audio_data = []
labels = []

for label_dir in os.listdir(data_path):
    label_path = os.path.join(data_path, label_dir)
    if os.path.isdir(label_path):
        for audio_file in os.listdir(label_path):
            if audio_file.endswith('.wav'):
                audio_data.append(os.path.join(label_path, audio_file))
                labels.append(label_dir)

# Create a DataFrame
df = pd.DataFrame({
    'filepath': audio_data,
    'label': labels
})

print(f'Total audio files: {len(df)}')
display(df.head())

Total audio files: 11802


,filepath,label
0,./keyword_spot/keyword_data/other/dog_225_5907...,other
1,./keyword_spot/keyword_data/other/noise_2_30.wav,other
2,./keyword_spot/keyword_data/other/zero_1895_26...,other
3,./keyword_spot/keyword_data/other/cat_190_695c...,other
4,./keyword_spot/keyword_data/other/dog_290_1a42...,other


Now, we will encode the categorical labels into numerical format and split the dataset into training and testing sets using `train_test_split`.

In [4]:
# Encode labels
le = LabelEncoder()
df['encoded_label'] = le.fit_transform(df['label'])

# Split data into training and testing sets
X = df['filepath']
y = df['encoded_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train set size: {len(X_train)}')
print(f'Test set size: {len(X_test)}')
print(f'Labels: {le.classes_}')

Train set size: 9441
Test set size: 2361
Labels: ['down' 'left' 'other' 'right' 'up']


For simple feature extraction, we'll convert each audio file into a Mel spectrogram. We need to ensure all spectrograms have a consistent shape. We'll define a function to process each audio file.

In [ ]:
import numpy as np
import scipy.io.wavfile as wav
from scipy.fftpack import dct
import os

# Configuration class for feature extraction
class FEConfig:
    def __init__(self, nfilt=40, nfft=512, target_time_steps=100):
        self.nfilt = nfilt             # Number of mel filter banks
        self.nfft = nfft               # FFT size
        self.target_time_steps = target_time_steps  # Padded time dimension

    def __repr__(self):
        return f"FEConfig(nfilt={self.nfilt}, nfft={self.nfft}, target_time_steps={self.target_time_steps})"

def extract_mel_spectrogram_raw(file_path, fe_config=None):
    if fe_config is None:
        fe_config = FEConfig()
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Không tìm thấy file: {file_path}")

    sample_rate, signal = wav.read(file_path)
    if signal.dtype == np.int16:
        signal = signal.astype(np.float32)
        
    pre_emphasis = 0.97
    emphasized_signal = np.append(signal[0], signal[1:] - pre_emphasis * signal[:-1])

    frame_size, frame_stride = 0.025, 0.01
    frame_length, frame_step = frame_size * sample_rate, frame_stride * sample_rate
    signal_length = len(emphasized_signal)
    frame_length = int(round(frame_length))
    frame_step = int(round(frame_step))
    num_frames = int(np.ceil(float(np.abs(signal_length - frame_length)) / frame_step))

    pad_signal_length = num_frames * frame_step + frame_length
    z = np.zeros((pad_signal_length - signal_length))
    pad_signal = np.append(emphasized_signal, z)

    indices = np.tile(np.arange(0, frame_length), (num_frames, 1)) + \
              np.tile(np.arange(0, num_frames * frame_step, frame_step), (frame_length, 1)).T
    frames = pad_signal[indices.astype(np.int32, copy=False)]
    frames *= np.hamming(frame_length)

    mag_frames = np.absolute(np.fft.rfft(frames, fe_config.nfft)) 
    pow_frames = ((1.0 / fe_config.nfft) * (mag_frames ** 2))

    low_freq_mel = 0
    high_freq_mel = (2595 * np.log10(1 + (sample_rate / 2) / 700))
    mel_points = np.linspace(low_freq_mel, high_freq_mel, fe_config.nfilt + 2)
    hz_points = (700 * (10**(mel_points / 2595) - 1))
    bin = np.floor((fe_config.nfft + 1) * hz_points / sample_rate)

    fbank = np.zeros((fe_config.nfilt, int(np.floor(fe_config.nfft / 2 + 1))))
    for m in range(1, fe_config.nfilt + 1):
        f_m_minus = int(bin[m - 1])
        f_m = int(bin[m])
        f_m_plus = int(bin[m + 1])
        for k in range(f_m_minus, f_m):
            fbank[m - 1, k] = (k - bin[m - 1]) / (bin[m] - bin[m - 1])
        for k in range(f_m, f_m_plus):
            fbank[m - 1, k] = (bin[m + 1] - k) / (bin[m + 1] - bin[m])
    
    mel_spectrogram = np.dot(pow_frames, fbank.T)
    mel_spectrogram = np.where(mel_spectrogram == 0, np.finfo(float).eps, mel_spectrogram)
    mel_spectrogram = 20 * np.log10(mel_spectrogram)

    current_frames = mel_spectrogram.shape[0]
    if current_frames < fe_config.target_time_steps:
        pad_width = ((0, fe_config.target_time_steps - current_frames), (0, 0))
        mel_padded = np.pad(mel_spectrogram, pad_width, mode='constant', constant_values=0)
    else:
        indices = np.linspace(0, current_frames - 1, fe_config.target_time_steps).astype(int)
        mel_padded = mel_spectrogram[indices]

    return mel_padded

all_train_features = []
current_fe_config = FEConfig(nfilt=40, nfft=512, target_time_steps=100)

print(f"Đang trích xuất đặc trưng từ {len(X_train)} file để tính toán chuẩn hóa...")

for filepath in X_train:
    try:
        mel = extract_mel_spectrogram_raw(filepath, current_fe_config)
        all_train_features.append(mel)
    except Exception as e:
        continue

all_train_features = np.array(all_train_features)

global_mean = np.mean(all_train_features, axis=(0, 1))
global_std = np.std(all_train_features, axis=(0, 1))

def extract_mel_spectrogram(file_path, fe_config=None, global_mean=global_mean, global_std=global_std):
    if fe_config is None:
        fe_config = FEConfig()
    
    # 1. Load and Pre-emphasis
    sample_rate, signal = wav.read(file_path)
    
    if signal.dtype == np.int16:
        signal = signal.astype(np.float32)
        
    pre_emphasis = 0.97
    emphasized_signal = np.append(signal[0], signal[1:] - pre_emphasis * signal[:-1])

    # 2. Framing
    frame_size, frame_stride = 0.025, 0.01  # 25ms and 10ms
    frame_length, frame_step = frame_size * sample_rate, frame_stride * sample_rate
    signal_length = len(emphasized_signal)
    frame_length = int(round(frame_length))
    frame_step = int(round(frame_step))
    num_frames = int(np.ceil(float(np.abs(signal_length - frame_length)) / frame_step))

    pad_signal_length = num_frames * frame_step + frame_length
    z = np.zeros((pad_signal_length - signal_length))
    pad_signal = np.append(emphasized_signal, z)

    indices = np.tile(np.arange(0, frame_length), (num_frames, 1)) + \
              np.tile(np.arange(0, num_frames * frame_step, frame_step), (frame_length, 1)).T
    frames = pad_signal[indices.astype(np.int32, copy=False)]

    # 3. Windowing (Hamming)
    frames *= np.hamming(frame_length)

    # 4. FFT and Power Spectrum
    mag_frames = np.absolute(np.fft.rfft(frames, fe_config.nfft)) 
    pow_frames = ((1.0 / fe_config.nfft) * (mag_frames ** 2))

    # 5. Filter Banks
    low_freq_mel = 0
    high_freq_mel = (2595 * np.log10(1 + (sample_rate / 2) / 700))
    mel_points = np.linspace(low_freq_mel, high_freq_mel, fe_config.nfilt + 2)
    hz_points = (700 * (10**(mel_points / 2595) - 1))
    bin = np.floor((fe_config.nfft + 1) * hz_points / sample_rate)

    fbank = np.zeros((fe_config.nfilt, int(np.floor(fe_config.nfft / 2 + 1))))
    for m in range(1, fe_config.nfilt + 1):
        f_m_minus = int(bin[m - 1])
        f_m = int(bin[m])
        f_m_plus = int(bin[m + 1])
        for k in range(f_m_minus, f_m):
            fbank[m - 1, k] = (k - bin[m - 1]) / (bin[m] - bin[m - 1])
        for k in range(f_m, f_m_plus):
            fbank[m - 1, k] = (bin[m + 1] - k) / (bin[m + 1] - bin[m])
    
    filter_banks = np.dot(pow_frames, fbank.T)
    filter_banks = np.where(filter_banks == 0, np.finfo(float).eps, filter_banks)
    filter_banks = 20 * np.log10(filter_banks) # dB

    mel_spectrogram = (mel_spectrogram - global_mean) / (global_std + 1e-8)

    # 8. Pad/Interpolate to target time steps
    current_frames = mel_spectrogram.shape[0]
    if current_frames < fe_config.target_time_steps:
        pad_width = ((0, fe_config.target_time_steps - current_frames), (0, 0))
        mel_padded = np.pad(mel_spectrogram, pad_width, mode='constant', constant_values=0)
    else:
        indices = np.linspace(0, current_frames - 1, fe_config.target_time_steps).astype(int)
        mel_padded = mel_spectrogram[indices]

    return mel_padded  # Shape: (target_time_steps, num_ceps)

In [6]:
# # Extract features from all audio files

# # Initialize FE config
# fe_config = FEConfig(num_ceps=40, nfilt=40, nfft=512, target_time_steps=100)

# # Extract train features
# X_train_features = []
# for filepath in X_train:
#     try:
#         mfcc = extract_mfcc(filepath, fe_config)
#         X_train_features.append(mfcc)
#     except Exception as e:
#         print(f"Error processing {filepath}: {e}")

# X_train_features = np.array(X_train_features)

# # Extract test features
# X_test_features = []
# for filepath in X_test:
#     try:
#         mfcc = extract_mfcc(filepath, fe_config)
#         X_test_features.append(mfcc)
#     except Exception as e:
#         print(f"Error processing {filepath}: {e}")

# X_test_features = np.array(X_test_features)

# print(f'FE Config: {fe_config}')
# print(f'Shape of X_train_features: {X_train_features.shape}')
# print(f'Shape of y_train: {y_train.shape}')
# print(f'Shape of X_test_features: {X_test_features.shape}')
# print(f'Shape of y_test: {y_test.shape}')

In [7]:
# np.save("./keyword_spot/x_train_features", X_train_features)
# np.save("./keyword_spot/x_test_features", X_test_features)
# np.save("./keyword_spot/y_train", y_train.values)
# np.save("./keyword_spot/y_test", y_test.values)
# print("✓ Features saved successfully")

In [8]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np

# Configuration class for model
class ModelConfig:
    def __init__(self, input_shape=(100, 40, 1), num_classes=NUM_CLASSES, embed_dim=32, num_heads=NUM_CLASSES, ff_dim=64):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
    
    def __repr__(self):
        return f"ModelConfig(input_shape={self.input_shape}, num_classes={self.num_classes}, embed_dim={self.embed_dim}, num_heads={self.num_heads}, ff_dim={self.ff_dim})"

# # Auto-align model config with FE output
# # FE output: (time_steps=100, num_ceps=40) -> after adding channel: (100, 40, 1)
# model_config = ModelConfig(
#     input_shape=(fe_config.target_time_steps, fe_config.num_ceps, 1),
#     num_classes=NUM_CLASSES,
#     embed_dim=32,
#     num_heads=4,
#     ff_dim=64
# )

# print(f"Model Config: {model_config}")

I0000 00:00:1778387120.666473  151511 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778387121.268793  151511 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778387123.048731  151511 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [9]:
def build_kws_transformer_v2(model_config):
    inputs = layers.Input(shape=model_config.input_shape)

    # 1. Feature extraction and dimension reduction (important for large spectrograms)
    # Dynamically compute stride and padding based on input size
    freq_dim = model_config.input_shape[0]
    time_dim = model_config.input_shape[1]
    
    # Layer 1: Apply Conv2D with stride (2,2) to reduce dimensions
    x = layers.Conv2D(8, (3, 3), strides=(2, 2), padding="same", activation="relu")(inputs)
    freq_dim //= 2
    time_dim //= 2
    
    # Layer 2: Further reduce
    x = layers.Conv2D(16, (3, 3), strides=(2, 2), padding="same", activation="relu")(x)
    freq_dim //= 2
    time_dim //= 2

    # 2. Prepare sequence for Transformer
    # Reshape to (Batch, Time_Steps, Features)
    x = layers.Permute((2, 1, 3))(x)  # Swap time and freq dimensions
    x = layers.Reshape((time_dim, freq_dim * 16))(x)

    # 3. Project to embedding dimension
    x = layers.Dense(model_config.embed_dim)(x)

    # 4. Positional Encoding
    positions = tf.range(start=0, limit=time_dim, delta=1)
    pos_encoding = layers.Embedding(input_dim=time_dim, output_dim=model_config.embed_dim)(positions)
    x = x + pos_encoding

    # 5. Transformer Block
    attn_output = layers.MultiHeadAttention(num_heads=model_config.num_heads, key_dim=model_config.embed_dim)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attn_output)

    ffn_output = layers.Dense(model_config.ff_dim, activation="relu")(x)
    ffn_output = layers.Dense(model_config.embed_dim)(ffn_output)
    x = layers.LayerNormalization(epsilon=1e-6)(x + ffn_output)

    # 6. Output
    x = layers.GlobalAveragePooling1D()(x)
    outputs = layers.Dense(model_config.num_classes, activation="softmax")(x)

    return tf.keras.Model(inputs=inputs, outputs=outputs)

In [10]:
# model = build_kws_transformer_v2(model_config)
# model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
# model.summary()

In [ ]:
# Tuning functions for FE and Model configuration

def tune_fe_config(
    num_ceps=None,
    nfilt=None,
    nfft=None,
    target_time_steps=None,
    X_train_data=None,
    X_test_data=None,
):
    """
    Tune feature extraction configuration.
    Returns updated FEConfig and extracted features.

    Args:
        num_ceps: Number of MFCCs (frequency bins). Default: current fe_config.num_ceps
        nfilt: Number of mel filter banks. Default: current fe_config.nfilt
        nfft: FFT size. Default: current fe_config.nfft
        target_time_steps: Fixed time dimension. Default: current fe_config.target_time_steps
        X_train_data: Raw training filepaths (before FE). Default: global X_train
        X_test_data: Raw test filepaths (before FE). Default: global X_test
    """
    global fe_config, X_train_features, X_test_features

    if X_train_data is None:
        X_train_data = X_train
    if X_test_data is None:
        X_test_data = X_test

    fe_config = FEConfig(
        num_ceps=num_ceps or fe_config.num_ceps,
        nfilt=nfilt or fe_config.nfilt,
        nfft=nfft or fe_config.nfft,
        target_time_steps=target_time_steps or fe_config.target_time_steps,
    )

    print(f"Updated FE Config: {fe_config}")

    # Re-extract features from raw paths
    print("Re-extracting train features from raw X_train data...")
    X_train_features = []
    for i, filepath in enumerate(X_train_data):
        try:
            mel_spectrogram = extract_mel_spectrogram(filepath, fe_config)
            X_train_features.append(mel_spectrogram)
        except Exception as e:
            print(f"Error processing {filepath}: {e}")

    X_train_features = np.array(X_train_features)

    print("Re-extracting test features from raw X_test data...")
    X_test_features = []
    for i, filepath in enumerate(X_test_data):
        try:
            mel_spectrogram = extract_mel_spectrogram(filepath, fe_config)
            X_test_features.append(mel_spectrogram)
        except Exception as e:
            print(f"Error processing {filepath}: {e}")

    X_test_features = np.array(X_test_features)

    print(f"New X_train_features shape: {X_train_features.shape}")
    print(f"New X_test_features shape: {X_test_features.shape}")

    return fe_config, X_train_features, X_test_features


def tune_model_config(embed_dim=None, num_heads=None, ff_dim=None):
    """
    Tune model configuration and rebuild model.

    Args:
        embed_dim: Embedding dimension. Default: 32
        num_heads: Number of attention heads. Default: 4
        ff_dim: Feed-forward dimension. Default: 64
    """
    global model_config, model

    model_config = ModelConfig(
        input_shape=(fe_config.target_time_steps, fe_config.num_ceps, 1),
        num_classes=NUM_CLASSES,
        embed_dim=embed_dim or model_config.embed_dim,
        num_heads=num_heads or model_config.num_heads,
        ff_dim=ff_dim or model_config.ff_dim,
    )

    print(f"Updated Model Config: {model_config}")

    # Rebuild model
    model = build_kws_transformer_v2(model_config)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    print("Model rebuilt and compiled")

    return model_config, model

In [12]:

# ============================================================
# TRAINING CONFIGURATION & PIPELINE
# ============================================================

class TrainingConfig:
    """Configuration for model training"""
    def __init__(self, epochs=15, batch_size=32, validation_split=0.2, 
                 early_stopping_patience=5, learning_rate=0.001):
        self.epochs = epochs
        self.batch_size = batch_size
        self.validation_split = validation_split
        self.early_stopping_patience = early_stopping_patience
        self.learning_rate = learning_rate
    
    def __repr__(self):
        return (f"TrainingConfig(epochs={self.epochs}, batch_size={self.batch_size}, "
                f"validation_split={self.validation_split}, "
                f"early_stopping_patience={self.early_stopping_patience}, "
                f"learning_rate={self.learning_rate})")


def train_model(X_train, y_train, X_test, y_test, model, train_config, 
                model_name="model", verbose=1):
    """
    Train model and return comprehensive results.
    
    Args:
        X_train: Training features (samples, time_steps, num_ceps)
        y_train: Training labels
        X_test: Test features
        y_test: Test labels
        model: Keras model to train
        train_config: TrainingConfig instance
        model_name: Name for tracking results
        verbose: Verbosity level
    
    Returns:
        Dict with results: {
            'model': trained model,
            'model_name': model name,
            'history': training history,
            'test_loss': test loss,
            'test_accuracy': test accuracy,
            'train_accuracy': final training accuracy,
            'y_pred': predictions on test set,
            'fe_config': current FE config,
            'model_config': current model config,
            'train_config': training config
        }
    """
    import time
    
    # Add channel dimension
    X_train_reshaped = X_train[..., np.newaxis]
    X_test_reshaped = X_test[..., np.newaxis]
    
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")
    print(f"X_train shape: {X_train_reshaped.shape}, y_train shape: {y_train.shape}")
    print(f"X_test shape: {X_test_reshaped.shape}, y_test shape: {y_test.shape}")
    print(f"Training Config: {train_config}")
    
    # Setup optimizer with learning rate
    optimizer = tf.keras.optimizers.Adam(learning_rate=train_config.learning_rate)
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    
    # Callbacks
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=train_config.early_stopping_patience,
            restore_best_weights=True,
            verbose=1
        )
    ]
    
    # Train
    start_time = time.time()
    history = model.fit(
        X_train_reshaped,
        y_train,
        epochs=train_config.epochs,
        batch_size=train_config.batch_size,
        validation_split=train_config.validation_split,
        callbacks=callbacks,
        verbose=verbose
    )
    training_time = time.time() - start_time
    
    # Evaluate
    train_loss, train_accuracy = model.evaluate(X_train_reshaped, y_train, verbose=0)
    test_loss, test_accuracy = model.evaluate(X_test_reshaped, y_test, verbose=0)
    
    # Predictions
    y_pred_probs = model.predict(X_test_reshaped, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    
    print(f"\nTraining completed in {training_time:.2f}s")
    print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}")
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")
    
    return {
        'model': model,
        'model_name': model_name,
        'history': history,
        'test_loss': test_loss,
        'test_accuracy': test_accuracy,
        'train_loss': train_loss,
        'train_accuracy': train_accuracy,
        'training_time': training_time,
        'y_pred': y_pred,
        'y_pred_probs': y_pred_probs,
        'fe_config': fe_config,
        'model_config': model_config,
        'train_config': train_config
    }


# Initialize default training config
train_config = TrainingConfig(
    epochs=15,
    batch_size=32,
    validation_split=0.2,
    early_stopping_patience=5,
    learning_rate=0.001
)

print(f"Default Training Config: {train_config}")


Default Training Config: TrainingConfig(epochs=15, batch_size=32, validation_split=0.2, early_stopping_patience=5, learning_rate=0.001)


In [13]:
# ============================================================
# FE CONFIGURATION TUNING PIPELINE
# ============================================================

# Define FE configs to test
fe_configs_to_test = [
    # (num_ceps, nfilt, nfft, target_time_steps, config_name)
    (32, 32, 512, 80, "FE_Config_1: Smaller MFCCs"),
    (40, 40, 512, 100, "FE_Config_2: Default"),
    (48, 48, 512, 100, "FE_Config_3: Larger MFCCs"),
    (40, 40, 256, 100, "FE_Config_4: Smaller FFT"),
    (40, 40, 1024, 100, "FE_Config_5: Larger FFT"),
    (40, 40, 512, 120, "FE_Config_6: More time steps"),
    (64, 64, 512, 100, "FE_Config_7: Large MFCCs+Filters"),
]

# Define Model configs to test (will use best FE config)
model_configs_to_test = [
    # (embed_dim, num_heads, ff_dim, config_name)
    (32, 4, 64, "Model_Config_1: Default"),
    (64, 8, 128, "Model_Config_2: Larger"),
    (16, 2, 32, "Model_Config_3: Smaller"),
]


def tune_fe_pipeline(X_train_data, y_train_data, X_test_data, y_test_data,
                     fe_configs, model_config_dict, train_config_dict, verbose=0):
    """
    Test multiple FE configurations and return results.

    Args:
        X_train_data, y_train_data: Raw training data and labels
        X_test_data, y_test_data: Raw test data and labels
        fe_configs: List of tuples (num_ceps, nfilt, nfft, target_time_steps, name)
        model_config_dict: Dict with model parameters
        train_config_dict: Dict with training parameters
        verbose: Verbosity level

    Returns:
        List of result dicts for each FE config
    """
    from sklearn.metrics import accuracy_score

    fe_results = []

    for i, (num_ceps, nfilt, nfft, target_time_steps, config_name) in enumerate(fe_configs):
        print(f"\n{'='*70}")
        print(f"[{i+1}/{len(fe_configs)}] Testing: {config_name}")
        print(f"{'='*70}")

        # Extract features from raw X_train/X_test with current FE config
        fe_cfg, X_tr_feat, X_te_feat = tune_fe_config(
            num_ceps=num_ceps,
            nfilt=nfilt,
            nfft=nfft,
            target_time_steps=target_time_steps,
            X_train_data=X_train_data,
            X_test_data=X_test_data,
        )

        # Update model config to match FE output
        mod_cfg, mod = tune_model_config(**model_config_dict)

        mod.summary()

        # Train model
        train_cfg = TrainingConfig(**train_config_dict)
        result = train_model(
            X_tr_feat, y_train_data,
            X_te_feat, y_test_data,
            mod, train_cfg,
            model_name=config_name,
            verbose=verbose,
        )

        # Add config name and calculate accuracy
        result['config_name'] = config_name
        result['test_accuracy_score'] = accuracy_score(y_test_data, result['y_pred'])

        fe_results.append(result)

        print(f"Test Accuracy: {result['test_accuracy']:.4f}")

    return fe_results


def compare_fe_results(fe_results, metric='test_accuracy'):
    """
    Compare FE configuration results and display summary.

    Args:
        fe_results: List of result dicts from tune_fe_pipeline
        metric: Metric to compare ('test_accuracy', 'train_accuracy', 'test_loss')

    Returns:
        Sorted dataframe by metric
    """
    import pandas as pd

    comparison_data = []
    for result in fe_results:
        comparison_data.append({
            'Config': result['config_name'],
            'Test Acc': result['test_accuracy'],
            'Train Acc': result['train_accuracy'],
            'Test Loss': result['test_loss'],
            'Train Loss': result['train_loss'],
            'Time (s)': result['training_time'],
            'FE Params': f"ceps={result['fe_config'].num_ceps}, "
                        f"nfilt={result['fe_config'].nfilt}, "
                        f"steps={result['fe_config'].target_time_steps}",
        })

    comparison_df = pd.DataFrame(comparison_data)

    if metric == 'test_accuracy':
        comparison_df = comparison_df.sort_values('Test Acc', ascending=False)
    elif metric == 'test_loss':
        comparison_df = comparison_df.sort_values('Test Loss', ascending=True)

    print(f"\n{'='*100}")
    print(f"FE Configuration Comparison (sorted by {metric})")
    print(f"{'='*100}")
    print(comparison_df.to_string(index=False))
    print(f"{'='*100}\n")

    return comparison_df


# Save results to file
def save_results(fe_results, save_path="./fe_tuning_results.txt"):
    """Save tuning results to file"""
    with open(save_path, 'w') as f:
        for result in fe_results:
            f.write(f"\n{'='*60}\n")
            f.write(f"Config: {result['config_name']}\n")
            f.write(f"{'='*60}\n")
            f.write(f"FE Config: {result['fe_config']}\n")
            f.write(f"Model Config: {result['model_config']}\n")
            f.write(f"Train Config: {result['train_config']}\n")
            f.write(f"Test Accuracy: {result['test_accuracy']:.6f}\n")
            f.write(f"Test Loss: {result['test_loss']:.6f}\n")
            f.write(f"Train Accuracy: {result['train_accuracy']:.6f}\n")
            f.write(f"Train Loss: {result['train_loss']:.6f}\n")
            f.write(f"Training Time: {result['training_time']:.2f}s\n")

    print(f"Results saved to {save_path}")


# Prepare training configs for tuning
default_model_params = {
    'embed_dim': 32,
    'num_heads': 4,
    'ff_dim': 64,
}

default_train_params = {
    'epochs': 15,
    'batch_size': 32,
    'validation_split': 0.2,
    'early_stopping_patience': 5,
    'learning_rate': 0.001,
}

# RUN FE TUNING (enable TUNE_FE flag to execute)
if TUNE_FE:
    print("\n" + "="*70)
    print("STARTING FE CONFIGURATION TUNING")
    print("="*70)

    fe_tuning_results = tune_fe_pipeline(
        X_train_data=X_train,
        y_train_data=y_train.values,
        X_test_data=X_test,
        y_test_data=y_test.values,
        fe_configs=fe_configs_to_test,
        model_config_dict=default_model_params,
        train_config_dict=default_train_params,
        verbose=0,
    )

    comparison_results = compare_fe_results(fe_tuning_results, metric='test_accuracy')
    save_results(fe_tuning_results, save_path="./keyword_spot/fe_tuning_results.txt")

    best_result = max(fe_tuning_results, key=lambda x: x['test_accuracy'])
    print(f"\nBEST FE CONFIG: {best_result['config_name']}")
    print(f"  Test Accuracy: {best_result['test_accuracy']:.4f}")
    print(f"  Training Time: {best_result['training_time']:.2f}s")
else:
    print("TUNE_FE=False, skipped FE tuning")


STARTING FE CONFIGURATION TUNING

[1/7] Testing: FE_Config_1: Smaller MFCCs
Updated FE Config: FEConfig(num_ceps=32, nfilt=32, nfft=512, target_time_steps=80)
Re-extracting train features from raw X_train data...
Re-extracting test features from raw X_test data...
New X_train_features shape: (9441, 80, 32)
New X_test_features shape: (2361, 80, 32)
Updated Model Config: ModelConfig(input_shape=(80, 32, 1), num_classes=5, embed_dim=32, num_heads=4, ff_dim=64)
Model rebuilt and compiled


W0000 00:00:1778387140.475421  151511 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 80, 32, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 40, 16, 8) │         80 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 20, 8, 16) │      1,168 │ conv2d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ permute (Permute)   │ (None, 8, 20, 16) │          0 │ conv2d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 8, 320)    │          0 │ permute[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 8, 32)     │     10,272 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 8, 32)     │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 8, 32)     │     16,800 │ add[0][0],        │
│ (MultiHeadAttentio… │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 8, 32)     │          0 │ add[0][0],        │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 8, 32)     │         64 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 8, 64)     │      2,112 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 8, 32)     │      2,080 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 8, 32)     │          0 │ layer_normalizat… │
│                     │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 8, 32)     │         64 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 5)         │        165 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,805 (128.14 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 100, 64,   │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_12 (Conv2D)  │ (None, 50, 32, 8) │         80 │ input_layer_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_13 (Conv2D)  │ (None, 25, 16,    │      1,168 │ conv2d_12[0][0]   │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ permute_6 (Permute) │ (None, 16, 25,    │          0 │ conv2d_13[0][0]   │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_6 (Reshape) │ (None, 16, 400)   │          0 │ permute_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_24 (Dense)    │ (None, 16, 32)    │     12,832 │ reshape_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_18 (Add)        │ (None, 16, 32)    │          0 │ dense_24[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 16, 32)    │     16,800 │ add_18[0][0],     │
│ (MultiHeadAttentio… │                   │            │ add_18[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_19 (Add)        │ (None, 16, 32)    │          0 │ add_18[0][0],     │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 16, 32)    │         64 │ add_19[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_25 (Dense)    │ (None, 16, 64)    │      2,112 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_26 (Dense)    │ (None, 16, 32)    │      2,080 │ dense_25[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_20 (Add)        │ (None, 16, 32)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_26[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 16, 32)    │         64 │ add_20[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_27 (Dense)    │ (None, 5)         │        165 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 35,365 (138.14 KB)

 Trainable params: 35,365 (138.14 KB)

 Non-trainable params: 0 (0.00 B)


Training: FE_Config_7: Large MFCCs+Filters
X_train shape: (9441, 100, 64, 1), y_train shape: (9441,)
X_test shape: (2361, 100, 64, 1), y_test shape: (2361,)
Training Config: TrainingConfig(epochs=15, batch_size=32, validation_split=0.2, early_stopping_patience=5, learning_rate=0.001)
Epoch 15: early stopping
Restoring model weights from the end of the best epoch: 10.

Training completed in 26.32s
Train Loss: 0.3299, Train Accuracy: 0.8871
Test Loss: 0.4910, Test Accuracy: 0.8357
Test Accuracy: 0.8357

FE Configuration Comparison (sorted by test_accuracy)
                          Config  Test Acc  Train Acc  Test Loss  Train Loss  Time (s)                    FE Params
    FE_Config_6: More time steps  0.857264   0.907425   0.422691    0.271385 24.960773 ceps=40, nfilt=40, steps=120
       FE_Config_3: Larger MFCCs  0.850064   0.905095   0.429830    0.272080 26.975506 ceps=48, nfilt=48, steps=100
      FE_Config_1: Smaller MFCCs  0.847946   0.907743   0.442711    0.272355 24.571101  ce

In [ ]:
# ============================================================
# MODEL CONFIGURATION TUNING PIPELINE (After FE tuning)
# ============================================================

def tune_model_pipeline(
    X_train_data,
    y_train_data,
    X_test_data,
    y_test_data,
    fe_config_for_model,
    model_configs,
    train_config_dict,
    verbose=0,
):
    """
    Test multiple Model configurations with fixed FE config.

    Args:
        X_train_data, y_train_data: Raw training data (before FE) and labels
        X_test_data, y_test_data: Raw test data (before FE) and labels
        fe_config_for_model: FEConfig to use for extracting features
        model_configs: List of tuples (embed_dim, num_heads, ff_dim, name)
        train_config_dict: Dict with training parameters
        verbose: Verbosity level

    Returns:
        List of result dicts for each model config
    """
    from sklearn.metrics import accuracy_score

    print("Extracting features from raw X_train/X_test for model tuning...")
    X_train_features = np.array([extract_mel_spectrogram(fp, fe_config_for_model) for fp in X_train_data])
    X_test_features = np.array([extract_mel_spectrogram(fp, fe_config_for_model) for fp in X_test_data])

    print(f"X_train_features shape: {X_train_features.shape}")
    print(f"X_test_features shape: {X_test_features.shape}")

    model_results = []

    for i, (embed_dim, num_heads, ff_dim, config_name) in enumerate(model_configs):
        print(f"\n{'='*70}")
        print(f"[{i+1}/{len(model_configs)}] Testing: {config_name}")
        print(f"{'='*70}")

        mod_cfg, mod = tune_model_config(
            embed_dim=embed_dim,
            num_heads=num_heads,
            ff_dim=ff_dim,
        )

        train_cfg = TrainingConfig(**train_config_dict)
        result = train_model(
            X_train_features, y_train_data,
            X_test_features, y_test_data,
            mod, train_cfg,
            model_name=config_name,
            verbose=verbose,
        )

        result['config_name'] = config_name
        result['test_accuracy_score'] = accuracy_score(y_test_data, result['y_pred'])

        model_results.append(result)

        print(f"Test Accuracy: {result['test_accuracy']:.4f}")

    return model_results


def compare_model_results(model_results, metric='test_accuracy'):
    """Compare model configuration results"""
    import pandas as pd

    comparison_data = []
    for result in model_results:
        comparison_data.append({
            'Config': result['config_name'],
            'Test Acc': result['test_accuracy'],
            'Train Acc': result['train_accuracy'],
            'Test Loss': result['test_loss'],
            'Train Loss': result['train_loss'],
            'Time (s)': result['training_time'],
            'Model Params': f"dim={result['model_config'].embed_dim}, "
                           f"heads={result['model_config'].num_heads}, "
                           f"ff={result['model_config'].ff_dim}",
        })

    comparison_df = pd.DataFrame(comparison_data)

    if metric == 'test_accuracy':
        comparison_df = comparison_df.sort_values('Test Acc', ascending=False)
    elif metric == 'test_loss':
        comparison_df = comparison_df.sort_values('Test Loss', ascending=True)

    print(f"\n{'='*100}")
    print(f"Model Configuration Comparison (sorted by {metric})")
    print(f"{'='*100}")
    print(comparison_df.to_string(index=False))
    print(f"{'='*100}\n")

    return comparison_df


# RUN MODEL TUNING (after selecting best FE config)
if TUNE_MODEL and 'fe_tuning_results' in locals() and len(fe_tuning_results) > 0:
    print("\n" + "="*70)
    print("STARTING MODEL CONFIGURATION TUNING")
    print("="*70)

    best_fe_result = max(fe_tuning_results, key=lambda x: x['test_accuracy'])
    best_fe_cfg = best_fe_result['fe_config']
    print(f"\nUsing best FE config: {best_fe_result['config_name']}")

    model_tuning_results = tune_model_pipeline(
        X_train_data=X_train,
        y_train_data=y_train.values,
        X_test_data=X_test,
        y_test_data=y_test.values,
        fe_config_for_model=best_fe_cfg,
        model_configs=model_configs_to_test,
        train_config_dict=default_train_params,
        verbose=0,
    )

    model_comparison_results = compare_model_results(model_tuning_results, metric='test_accuracy')

    best_model_result = max(model_tuning_results, key=lambda x: x['test_accuracy'])
    print(f"\nBEST MODEL CONFIG: {best_model_result['config_name']}")
    print(f"  Test Accuracy: {best_model_result['test_accuracy']:.4f}")
else:
    print("TUNE_MODEL=False or no fe_tuning_results, skipped model tuning")

TUNE_MODEL=False or no fe_tuning_results, skipped model tuning


In [15]:
    # --- 3. Train model ---
if TRAIN:
    # Add channel dimension to match model input
    # X_train_features shape: (num_samples, time_steps, num_ceps)
    # Model expects: (num_samples, time_steps, num_ceps, 1)
    X_train_reshaped = X_train_features[..., np.newaxis]
    X_test_reshaped = X_test_features[..., np.newaxis]

    y_train_final = y_train.values
    y_test_final = y_test.values

    print(f"Shape of X_train_reshaped: {X_train_reshaped.shape}")
    print(f"Shape of y_train_final: {y_train_final.shape}")
    print(f"Shape of X_test_reshaped: {X_test_reshaped.shape}")
    print(f"Shape of y_test_final: {y_test_final.shape}")

    print("\nStarting model training...")
    history = model.fit(
        X_train_reshaped,
        y_train_final,
        epochs=15,
        batch_size=32,
        validation_data=(X_test_reshaped, y_test_final),
        callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
        verbose=1
    )

    print("\nEvaluating the model on the test set...")
    loss, accuracy = model.evaluate(X_test_reshaped, y_test_final, verbose=0)
    print(f"Test Loss: {loss:.4f}")
    print(f"Test Accuracy: {accuracy:.4f}")

    # Plot training history
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [16]:
    # 1. Chuyển sang TFLite với tối ưu hóa kích thước
if TRAIN:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT] # Quantization 8-bit weights
    tflite_model = converter.convert()

    # 2. Hàm ghi file .h chuyên dụng cho C++
    def export_to_header(tflite_model, c_array_name="kws_model"):
        with open(f"{c_array_name}.h", "w") as f:
            f.write(f"#ifndef {c_array_name.upper()}_H\n")
            f.write(f"#define {c_array_name.upper()}_H\n\n")
            f.write(f"const unsigned char {c_array_name}[] __attribute__((aligned(16))) = {{")

            for i, byte in enumerate(tflite_model):
                if i % 12 == 0: f.write("\n  ")
                f.write(f"0x{byte:02x}, ")

            f.write("\n};\n\n")
            f.write(f"const int {c_array_name}_len = {len(tflite_model)};\n")
            f.write(f"#endif\n")

    export_to_header(tflite_model, "tiny_kws_transformer")
    print("Đã tạo file tiny_kws_transformer.h")